<a href="https://colab.research.google.com/github/raki-rankawat/stm32-thesis/blob/main/MobileNetV2_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [106]:
# =====================================================
# Install ONNX / ONNXRuntime Tools
# =====================================================
!pip -q install onnx onnxruntime onnxscript onnxruntime-tools

In [107]:
# =====================================================
# Imports
# =====================================================

import os
import time
import tarfile
import random
import shutil
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import onnx
import onnxruntime as ort

from onnxruntime.quantization import (
    quantize_static,
    QuantType,
    QuantFormat,
    CalibrationDataReader
)

from google.colab import drive

In [108]:
# =====================================================
# Mount Google Drive
# =====================================================

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [109]:
# =====================================================
# Device Setup
# =====================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(41)
np.random.seed(41)
torch.manual_seed(41)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(41)
    torch.backends.cudnn.benchmark = True

print("Using device:", device)

Using device: cuda


### Prepare VWW

In [110]:
# =====================================================
# Auto Download + Prepare VWW (10k subset)
# =====================================================

VWW_URL = "https://www.silabs.com/public/files/github/machine_learning/benchmarks/datasets/vw_coco2014_96.tar.gz"

BASE_DIR = Path("/content/vww_work")
ARCHIVE_PATH = BASE_DIR / "vw_coco2014_96.tar.gz"
EXTRACT_DIR = BASE_DIR / "extracted"
SUBSET_DIR = BASE_DIR / "vww_10k"

N_PER_CLASS = 5000
VAL_RATIO = 0.20

def download_vww():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    if ARCHIVE_PATH.exists() and ARCHIVE_PATH.stat().st_size > 0:
        print("✅ VWW archive already downloaded")
        return

    print("⬇️ Downloading VWW archive...")
    urlretrieve(VWW_URL, ARCHIVE_PATH)
    print("✅ Download complete:", ARCHIVE_PATH)

def safe_extract_tar(tar, path="."):
    path = Path(path).resolve()

    for member in tar.getmembers():
        member_path = (path / member.name).resolve()
        if not str(member_path).startswith(str(path)):
            raise RuntimeError("❌ Unsafe path detected in tar archive")

    tar.extractall(path)

def extract_vww():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    if any(EXTRACT_DIR.iterdir()):
        print("✅ VWW already extracted")
        return

    print("📦 Extracting VWW archive...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        safe_extract_tar(tar, EXTRACT_DIR)

    print("✅ Extraction complete:", EXTRACT_DIR)

def find_vww_root():
    for p in EXTRACT_DIR.rglob("person"):
        if p.is_dir() and (p.parent / "non_person").is_dir():
            return p.parent

    raise RuntimeError("❌ Could not find 'person' and 'non_person' directories")

def list_images(folder):
    exts = {".jpg", ".jpeg", ".png"}
    return [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in exts]

def make_vww_subset(src_root):
    if (
        (SUBSET_DIR / "train" / "person").is_dir() and
        (SUBSET_DIR / "train" / "non_person").is_dir() and
        (SUBSET_DIR / "val" / "person").is_dir() and
        (SUBSET_DIR / "val" / "non_person").is_dir()
    ):
        print("✅ VWW 10k subset already exists:", SUBSET_DIR)
        return

    for split in ["train", "val"]:
        for c in ["person", "non_person"]:
            (SUBSET_DIR / split / c).mkdir(parents=True, exist_ok=True)

    person_imgs = list_images(src_root / "person")
    nonperson_imgs = list_images(src_root / "non_person")

    if len(person_imgs) < N_PER_CLASS or len(nonperson_imgs) < N_PER_CLASS:
        raise ValueError(
            f"❌ Not enough images:\n"
            f"person: {len(person_imgs)} (need {N_PER_CLASS})\n"
            f"non_person: {len(nonperson_imgs)} (need {N_PER_CLASS})"
        )

    random.shuffle(person_imgs)
    random.shuffle(nonperson_imgs)

    person_sel = person_imgs[:N_PER_CLASS]
    nonperson_sel = nonperson_imgs[:N_PER_CLASS]

    def split_list(lst):
        n_val = int(len(lst) * VAL_RATIO)
        return lst[n_val:], lst[:n_val]   # train, val

    p_train, p_val = split_list(person_sel)
    n_train, n_val = split_list(nonperson_sel)

    def copy_files(files, dst_dir):
        for f in files:
            dst = dst_dir / f.name
            if dst.exists():
                dst = dst_dir / f"{f.parent.name}_{f.name}"
            shutil.copy2(f, dst)

    print("🧩 Creating VWW 10k subset...")
    copy_files(p_train, SUBSET_DIR / "train" / "person")
    copy_files(p_val,   SUBSET_DIR / "val"   / "person")
    copy_files(n_train, SUBSET_DIR / "train" / "non_person")
    copy_files(n_val,   SUBSET_DIR / "val"   / "non_person")
    print("✅ VWW subset created at:", SUBSET_DIR)

download_vww()
extract_vww()

vww_root = find_vww_root()
print("✅ Found VWW root:", vww_root)

make_vww_subset(vww_root)

✅ VWW archive already downloaded
✅ VWW already extracted
✅ Found VWW root: /content/vww_work/extracted/vw_coco2014_96
✅ VWW 10k subset already exists: /content/vww_work/vww_10k


In [111]:
# =====================================================
# Dataset for Evaluation + Calibration
# =====================================================

IMG_SIZE = 96
EVAL_SAMPLES = 200
CALIB_SAMPLES = 200

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    )
])

val_dataset = datasets.ImageFolder(
    root=str(SUBSET_DIR / "val"),
    transform=eval_transform
)

train_dataset_for_calib = datasets.ImageFolder(
    root=str(SUBSET_DIR / "train"),
    transform=eval_transform
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

calib_loader = DataLoader(
    train_dataset_for_calib,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("Class mapping:", val_dataset.class_to_idx)
print("Validation samples:", len(val_dataset))
print("Training samples for calibration:", len(train_dataset_for_calib))

Class mapping: {'non_person': 0, 'person': 1}
Validation samples: 2000
Training samples for calibration: 8000


### MobileNetV2 Model

In [112]:
# =====================================================
# MobileNetV2 Block
# =====================================================

class InvertedResidual(nn.Module):

    def __init__(self, in_channels, out_channels, stride, expand_ratio):
        super().__init__()

        hidden_dim = in_channels * expand_ratio
        self.use_residual = stride == 1 and in_channels == out_channels

        self.block = nn.Sequential(

            nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),

            nn.Conv2d(
                hidden_dim,
                hidden_dim,
                3,
                stride=stride,
                padding=1,
                groups=hidden_dim,
                bias=False
            ),

            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),

            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):

        if self.use_residual:
            return x + self.block(x)

        return self.block(x)

In [113]:
# =====================================================
# MobileNetV2 Model
# =====================================================

class VWW_MobileNetV2(nn.Module):

    def __init__(self):
        super().__init__()

        self.initial = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True)
        )

        self.features = nn.Sequential(
            InvertedResidual(32, 16, 1, 1),

            InvertedResidual(16, 24, 2, 6),
            InvertedResidual(24, 24, 1, 6),

            InvertedResidual(24, 32, 2, 6),
            InvertedResidual(32, 32, 1, 6),

            InvertedResidual(32, 64, 2, 6),
            InvertedResidual(64, 64, 1, 6),
        )

        self.final_conv = nn.Sequential(
            nn.Conv2d(64, 512, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU6(inplace=True)
        )

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(512, 2)
        self._initialize_weights()

    def forward(self, x):
        x = self.initial(x)
        x = self.features(x)
        x = self.final_conv(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

    def _initialize_weights(self):

        for m in self.modules():

            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')

            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

In [114]:
# =====================================================
# Load Trained MobileNet Weights
# =====================================================

checkpoint_path = "/content/drive/My Drive/Colab Notebooks/vww_mobilenetv2_model.pth"

model = VWW_MobileNetV2().to(device)
state_dict = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print("✅ Loaded:", checkpoint_path)
print("✅ Model device:", next(model.parameters()).device)

✅ Loaded: /content/drive/My Drive/Colab Notebooks/vww_mobilenetv2_model.pth
✅ Model device: cuda:0


In [115]:
# =====================================================
# PyTorch Accuracy on First N Validation Samples
# =====================================================

correct = 0
total = 0

with torch.no_grad():
    for i, (x, y) in enumerate(val_loader):
        if i >= EVAL_SAMPLES:
            break

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        out = model(x)
        pred = out.argmax(dim=1)

        correct += (pred == y).sum().item()
        total += y.size(0)

print(f"PyTorch accuracy on first {EVAL_SAMPLES}:", 100 * correct / total)

PyTorch accuracy on first 200: 83.5


### FP32 Pipeline

In [116]:
# =====================================================
# Export FP32 ONNX
# =====================================================

def export_onnx(model, onnx_path):
    model.eval()

    dummy = torch.randn(1, 3, 96, 96, device=device)

    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        input_names=["input"],
        output_names=["logits"],
        export_params=True,
        opset_version=18,
        do_constant_folding=True,
        dynamic_axes={
            "input": {0: "batch_size"},
            "logits": {0: "batch_size"}
        },
        dynamo=False
    )

    onnx.checker.check_model(onnx_path, full_check=False)
    print(f"✅ ONNX model saved to: {onnx_path}")

export_onnx(model, "vww_mobilenetv2_fp32.onnx")

/tmp/ipykernel_776/1677125830.py:10: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


✅ ONNX model saved to: vww_mobilenetv2_fp32.onnx


In [117]:
# =====================================================
# Save Validation Inputs + PyTorch Logits + Labels
# Keep NCHW for STM32 / ST Edge AI
# =====================================================

inputs_nchw = []
logits = []
labels = []

model.eval()

with torch.no_grad():
    for i, (x, y) in enumerate(val_loader):
        if i >= EVAL_SAMPLES:
            break

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        out = model(x)

        x_nchw = x.detach().cpu().numpy().astype(np.float32)      # (1,3,96,96)
        out_np = out.detach().cpu().numpy()[0].astype(np.float32) # (2,)

        inputs_nchw.append(x_nchw[0])   # (3,96,96)
        logits.append(out_np)           # (2,)
        labels.append(int(y.item()))

inputs_nchw = np.stack(inputs_nchw, axis=0)   # (N,3,96,96)
logits = np.stack(logits, axis=0)             # (N,2)
labels = np.array(labels, dtype=np.int32)     # (N,)

np.savez("vww_mobilenetv2_val_200_io.npz", input=inputs_nchw, logits=logits)
np.savez("vww_mobilenetv2_labels_200.npz", label=labels)

print("✅ Saved validation IO and labels")
print("Input shape:", inputs_nchw.shape)
print("Input min/max:", inputs_nchw.min(), inputs_nchw.max())

✅ Saved validation IO and labels
Input shape: (200, 3, 96, 96)
Input min/max: -2.117904 2.64


In [118]:
# =====================================================
# Compute Accuracy from STM32 Output NPZ
# =====================================================

def compute_accuracy(
    labels_npz_path,
    outputs_npz_path,
    output_key="c_outputs_1",
    num_classes=2,
    as_percentage=False
):
    labels = np.load(labels_npz_path)["label"].astype(np.int64)
    out = np.load(outputs_npz_path)

    raw = out[output_key]
    total_values = raw.size
    expected_values = len(labels) * num_classes

    print("Labels:", len(labels))
    print("Output raw shape:", raw.shape)
    print("Output raw size:", total_values)
    print("Expected size:", expected_values)

    if total_values != expected_values:
        raise ValueError(
            f"Size mismatch: output has {total_values} values, "
            f"but expected {expected_values} = {len(labels)} x {num_classes}. "
            f"Use matching input/label/output files from the same run."
        )

    logits = raw.reshape(len(labels), num_classes)
    pred = np.argmax(logits, axis=1)

    acc = (pred == labels).mean()
    return acc * 100 if as_percentage else acc

In [127]:
# =====================================================
# Example: STM32 Accuracy Check
# =====================================================

# Run this only after STM32 / ST Edge AI produces network_val_io.npz
acc = compute_accuracy(
    "vww_mobilenetv2_labels_200.npz",
    "network_val_io.npz",
    output_key="c_outputs_1",
    num_classes=2,
    as_percentage=True
)

print("STM32 accuracy:", acc)

Labels: 200
Output raw shape: (200, 1, 1, 2)
Output raw size: 400
Expected size: 400
STM32 accuracy: 83.5


### INT8 Pipeline

In [128]:
# =====================================================
# Calibration NPZ
# Use TRAIN split for calibration
# =====================================================

def make_calib_npz(dataset, N=200, out_path="vww_mobilenetv2_calib_200.npz"):
    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    xs = []

    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= N:
                break

            # No need to send calibration tensors to GPU just to save them.
            x_np = x.numpy().astype(np.float32)[0]   # (3,96,96)
            xs.append(x_np)

    xs = np.stack(xs, axis=0)
    np.savez(out_path, input=xs)

    print("✅ Saved calib:", out_path, xs.shape)
    return out_path

In [129]:
# =====================================================
# Calibration Reader
# =====================================================

class CalibReader(CalibrationDataReader):

    def __init__(self, npz_path, input_name="input"):
        self.x = np.load(npz_path)["input"].astype(np.float32)
        self.input_name = input_name
        self.i = 0

    def get_next(self):
        if self.i >= len(self.x):
            return None

        batch = self.x[self.i:self.i + 1]   # (1,3,96,96)
        self.i += 1
        return {self.input_name: batch}

In [130]:
# =====================================================
# Export INT8 QDQ ONNX
# =====================================================

def quantize_int8_qdq(
    fp32_onnx="vww_mobilenetv2_fp32.onnx",
    calib_npz="vww_mobilenetv2_calib_200.npz",
    int8_onnx="vww_mobilenetv2_int8_static_qdq.onnx"
):
    reader = CalibReader(calib_npz, input_name="input")

    quantize_static(
        model_input=fp32_onnx,
        model_output=int8_onnx,
        calibration_data_reader=reader,
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QInt8,
        weight_type=QuantType.QInt8,
        per_channel=True,
    )

    print("✅ Saved INT8:", int8_onnx)
    return int8_onnx

calib_npz = make_calib_npz(
    train_dataset_for_calib,
    N=CALIB_SAMPLES,
    out_path="vww_mobilenetv2_calib_200.npz"
)

quantize_int8_qdq(
    "vww_mobilenetv2_fp32.onnx",
    calib_npz,
    "vww_mobilenetv2_int8_static_qdq.onnx"
)

✅ Saved calib: vww_mobilenetv2_calib_200.npz (200, 3, 96, 96)


✅ Saved INT8: vww_mobilenetv2_int8_static_qdq.onnx


'vww_mobilenetv2_int8_static_qdq.onnx'

In [132]:
# =====================================================
# Optional: ONNX Runtime FP32 Check
# =====================================================

session = ort.InferenceSession(
    "vww_mobilenetv2_fp32.onnx",
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

sample_input = inputs_nchw[0:1].astype(np.float32)
ort_out = session.run([output_name], {input_name: sample_input})[0]

print("✅ ONNX Runtime output shape:", ort_out.shape)
print("Sample logits:", ort_out[0])

✅ ONNX Runtime output shape: (1, 2)
Sample logits: [ 0.8495252 -0.7710259]


In [134]:
# =====================================================
# Optional: Save Export Files to Drive
# =====================================================

export_dir = Path("/content/drive/My Drive/Colab Notebooks/vww_mobilenetv2_exports")
export_dir.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    "vww_mobilenetv2_fp32.onnx",
    "vww_mobilenetv2_int8_static_qdq.onnx",
    "vww_mobilenetv2_val_200_io.npz",
    "vww_mobilenetv2_labels_200.npz",
    "vww_mobilenetv2_calib_200.npz",
]

for file_name in files_to_copy:
    src = Path(file_name)
    if src.exists():
        shutil.copy2(src, export_dir / src.name)
        print(f"✅ Copied {src.name}")

print("✅ Export folder:", export_dir)

✅ Copied vww_mobilenetv2_fp32.onnx
✅ Copied vww_mobilenetv2_int8_static_qdq.onnx
✅ Copied vww_mobilenetv2_val_200_io.npz
✅ Copied vww_mobilenetv2_labels_200.npz
✅ Copied vww_mobilenetv2_calib_200.npz
✅ Export folder: /content/drive/My Drive/Colab Notebooks/vww_mobilenetv2_exports


In [135]:
acc = compute_accuracy(
    "vww_mobilenetv2_labels_200.npz",
    "network_val_io.npz",
    output_key="c_outputs_1",
    num_classes=2,
    as_percentage=True
)

print("STM32 accuracy after:", acc)

Labels: 200
Output raw shape: (200, 1, 1, 2)
Output raw size: 400
Expected size: 400
STM32 accuracy after: 87.0
